In [1]:
# -*- coding: utf-8 -*-


In [2]:
# TRAINING SCRIPT - Prediksi Kelayakan Air Minum (ADVANCED + HYBRID PIPELINE)


In [5]:
import os
import sys
import json
import warnings
import time

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# Algoritma Machine Learning
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Teknik Imbalance Data
from imblearn.over_sampling import ADASYN

warnings.filterwarnings('ignore')

In [6]:
# KONFIGURASI PATH


In [8]:
import os
import seaborn as sns
import matplotlib.pyplot as plt

# Mendefinisikan direktori utama secara manual agar kompatibel dengan Jupyter Notebook
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATASET_PATH = os.path.join(BASE_DIR, 'dataset', 'water_potability.csv')
MODELS_DIR = os.path.join(BASE_DIR, 'models')
IMAGES_DIR = os.path.join(BASE_DIR, 'static', 'images')

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("=" * 70)
print("  ADVANCED TRAINING PIPELINE - Prediksi Kelayakan Air Minum")
print("=" * 70)

  ADVANCED TRAINING PIPELINE - Prediksi Kelayakan Air Minum


In [9]:
# TAHAP 1: LOAD DATASET


In [10]:
print("\n[TAHAP 1] Loading Dataset...")
print("-" * 50)
df = pd.read_csv(DATASET_PATH)
print(f"  Jumlah Sampel : {df.shape[0]}")
print(f"  Jumlah Fitur  : {df.shape[1] - 1}")




[TAHAP 1] Loading Dataset...
--------------------------------------------------
  Jumlah Sampel : 3276
  Jumlah Fitur  : 9


In [11]:
# TAHAP 2: ADVANCED IMPUTATION (KNN IMPUTER)


In [12]:
print("\n[TAHAP 2] Analisis Missing Values & KNN Imputation...")
print("-" * 50)
fitur_kolom = [col for col in df.columns if col != 'Potability']
print("  Menggunakan KNNImputer(n_neighbors=5, weights='distance')...")
imputer = KNNImputer(n_neighbors=5, weights='distance')
df[fitur_kolom] = imputer.fit_transform(df[fitur_kolom])
print(f"  Imputasi KNN selesai. Total missing values: {df.isnull().sum().sum()}")




[TAHAP 2] Analisis Missing Values & KNN Imputation...
--------------------------------------------------
  Menggunakan KNNImputer(n_neighbors=5, weights='distance')...
  Imputasi KNN selesai. Total missing values: 0


In [ ]:
# TAHAP 3 & 4: PENGHAPUSAN DATA DUPLIKAT & OUTLIER


In [13]:
print("\n[TAHAP 3 & 4] Duplikat & Outlier (IQR Capping)...")
print("-" * 50)
df = df.drop_duplicates()
def handle_outliers_iqr(dataframe, columns):
    df_capped = dataframe.copy()
    for col in columns:
        Q1 = df_capped[col].quantile(0.25)
        Q3 = df_capped[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df_capped[col] = np.clip(df_capped[col], lower_bound, upper_bound)
    return df_capped
df_clean = handle_outliers_iqr(df, fitur_kolom)
print("  Capping IQR selesai!")




[TAHAP 3 & 4] Duplikat & Outlier (IQR Capping)...
--------------------------------------------------
  Capping IQR selesai!


In [ ]:
# TAHAP 5: TRAIN-TEST SPLIT


In [14]:
print("\n[TAHAP 5] Train-Test Split (80:20 Stratified)...")
print("-" * 50)
X = df_clean[fitur_kolom]
y = df_clean['Potability']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"  Data Latih : {X_train.shape[0]} sampel")
print(f"  Data Uji   : {X_test.shape[0]} sampel")




[TAHAP 5] Train-Test Split (80:20 Stratified)...
--------------------------------------------------
  Data Latih : 2620 sampel
  Data Uji   : 656 sampel


In [ ]:
# TAHAP 6: ADASYN OVERSAMPLING


In [15]:
print("\n[TAHAP 6] ADASYN Oversampling (hanya pada data training)...")
print("-" * 50)
adasyn = ADASYN(random_state=42)
X_train_resampled, y_train_resampled = adasyn.fit_resample(X_train, y_train)
print(f"  Sebelum ADASYN - Tidak Layak: {(y_train==0).sum()} | Layak: {(y_train==1).sum()}")
print(f"  Sesudah ADASYN - Tidak Layak: {(y_train_resampled==0).sum()} | Layak: {(y_train_resampled==1).sum()}")




[TAHAP 6] ADASYN Oversampling (hanya pada data training)...
--------------------------------------------------
  Sebelum ADASYN - Tidak Layak: 1598 | Layak: 1022
  Sesudah ADASYN - Tidak Layak: 1598 | Layak: 1773


In [ ]:
# TAHAP 7: FEATURE SCALING (StandardScaler)


In [16]:
print("\n[TAHAP 7] Feature Scaling (StandardScaler)...")
print("-" * 50)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_resampled)
X_test_scaled = scaler.transform(X_test)
print("  StandardScaler berhasil diterapkan.")




[TAHAP 7] Feature Scaling (StandardScaler)...
--------------------------------------------------
  StandardScaler berhasil diterapkan.


In [17]:
# TAHAP 8: TRAINING & PERBANDINGAN MODEL (TERMASUK ENSEMBLE)


In [18]:
print("\n[TAHAP 8] Training & Perbandingan Model (dengan Class Weight & CatBoost)...")
print("-" * 50)

# Base strong models
rf_base = RandomForestClassifier(n_estimators=300, max_depth=20, min_samples_leaf=2, class_weight='balanced', random_state=42, n_jobs=-1)
xgb_base = XGBClassifier(n_estimators=300, max_depth=7, learning_rate=0.05, eval_metric='logloss', random_state=42, verbosity=0)
lgbm_base = LGBMClassifier(n_estimators=300, max_depth=7, learning_rate=0.05, class_weight='balanced', random_state=42, verbose=-1)
cb_base = CatBoostClassifier(iterations=300, depth=7, learning_rate=0.05, verbose=0, random_seed=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', random_state=42),
    'Random Forest': rf_base,
    'SVM (RBF)': SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42),
    'XGBoost': xgb_base,
    'LightGBM': lgbm_base,
    'CatBoost': cb_base,
    'Ensemble (RF+XGB+LGBM+CB)': VotingClassifier(
        estimators=[('rf', rf_base), ('xgb', xgb_base), ('lgbm', lgbm_base), ('cb', cb_base)],
        voting='soft', n_jobs=-1
    )
}

results = {}
skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f"\n  Training: {name}...")
    start_time = time.time()
    model.fit(X_train_scaled, y_train_resampled)
    
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
    elapsed = time.time() - start_time
    
    results[name] = {
        'model': model, 'accuracy': acc,
        'precision': precision_score(y_test, y_pred), 'recall': recall_score(y_test, y_pred),
        'f1_score': f1, 'roc_auc': roc, 'y_pred': y_pred, 'y_prob': y_prob,
        'training_time': elapsed
    }
    print(f"    Test Acc:{acc:.4f} | Recall:{results[name]['recall']:.4f} | F1:{f1:.4f} | ROC:{roc:.4f} | {elapsed:.1f}s")




[TAHAP 8] Training & Perbandingan Model (dengan Class Weight & CatBoost)...
--------------------------------------------------

  Training: Logistic Regression...
    Test Acc:0.5290 | Recall:0.5547 | F1:0.4789 | ROC:0.5384 | 0.0s

  Training: Decision Tree...
    Test Acc:0.5671 | Recall:0.5625 | F1:0.5035 | ROC:0.5663 | 0.1s

  Training: Random Forest...
    Test Acc:0.6159 | Recall:0.5039 | F1:0.5059 | ROC:0.6548 | 1.3s

  Training: SVM (RBF)...
    Test Acc:0.5930 | Recall:0.5352 | F1:0.5065 | ROC:0.6391 | 3.3s

  Training: XGBoost...
    Test Acc:0.6296 | Recall:0.5703 | F1:0.5458 | ROC:0.6590 | 0.5s

  Training: LightGBM...
    Test Acc:0.6082 | Recall:0.5312 | F1:0.5142 | ROC:0.6326 | 2.3s

  Training: CatBoost...
    Test Acc:0.6067 | Recall:0.5547 | F1:0.5240 | ROC:0.6556 | 2.3s

  Training: Ensemble (RF+XGB+LGBM+CB)...
    Test Acc:0.6052 | Recall:0.5156 | F1:0.5048 | ROC:0.6552 | 7.7s


In [ ]:
# TAHAP 9: PEMILIHAN MODEL TERBAIK & THRESHOLD TUNING


In [19]:
print("\n\n[TAHAP 9] Pemilihan Model (berdasarkan Recall & F1) & Threshold Tuning...")
print("-" * 50)

# Pilih model dengan bobot kombinasi F1 dan Recall tertinggi
best_model_name = max(results, key=lambda x: (results[x]['f1_score'] * 0.7) + (results[x]['recall'] * 0.3))
best_result = results[best_model_name]
final_model = best_result['model']
y_prob_best = best_result['y_prob']

print(f"  MODEL TERBAIK SEBELUM TUNING THRESHOLD: {best_model_name}")
print(f"  F1 Score (Threshold 0.50) : {best_result['f1_score']:.4f}")
print(f"  Recall (Threshold 0.50)   : {best_result['recall']:.4f}")

# Threshold Tuning untuk memaksimalkan Recall dan F1
print("\n  Mencari Optimal Decision Threshold...")
thresholds = np.arange(0.30, 0.71, 0.01)
best_thresh = 0.50
best_thresh_f1 = best_result['f1_score']
best_thresh_acc = best_result['accuracy']
best_thresh_prec = best_result['precision']
best_thresh_rec = best_result['recall']
best_y_pred = best_result['y_pred']

for thresh in thresholds:
    y_pred_thresh = (y_prob_best >= thresh).astype(int)
    f1_t = f1_score(y_test, y_pred_thresh)
    if f1_t > best_thresh_f1:
        best_thresh_f1 = f1_t
        best_thresh = thresh
        best_thresh_acc = accuracy_score(y_test, y_pred_thresh)
        best_thresh_prec = precision_score(y_test, y_pred_thresh)
        best_thresh_rec = recall_score(y_test, y_pred_thresh)
        best_y_pred = y_pred_thresh

print(f"  Optimal Threshold Ditemukan : {best_thresh:.2f}")
print(f"  F1 Score (Tuned)            : {best_thresh_f1:.4f}")
print(f"  Recall (Tuned)              : {best_thresh_rec:.4f}")





[TAHAP 9] Pemilihan Model (berdasarkan Recall & F1) & Threshold Tuning...
--------------------------------------------------
  MODEL TERBAIK SEBELUM TUNING THRESHOLD: XGBoost
  F1 Score (Threshold 0.50) : 0.5458
  Recall (Threshold 0.50)   : 0.5703

  Mencari Optimal Decision Threshold...
  Optimal Threshold Ditemukan : 0.32
  F1 Score (Tuned)            : 0.5681
  Recall (Tuned)              : 0.7656


In [ ]:
# TAHAP 10: FEATURE IMPORTANCE


In [20]:
print("\n[TAHAP 10] Analisis Feature Importance...")
print("-" * 50)
feature_importances_dict = {}

try:
    if hasattr(final_model, 'feature_importances_'):
        importances = final_model.feature_importances_
    elif hasattr(final_model, 'estimators_'): # Jika Ensemble
        importances = np.mean([est.feature_importances_ for est in final_model.estimators_ if hasattr(est, 'feature_importances_')], axis=0)
    else:
        importances = np.zeros(len(fitur_kolom))
        
    indices = np.argsort(importances)[::-1]
    
    print("  Feature ranking:")
    for f in range(X.shape[1]):
        feature_name = fitur_kolom[indices[f]]
        importance_score = importances[indices[f]]
        feature_importances_dict[feature_name] = float(importance_score)
        print(f"    {f + 1}. {feature_name} ({importance_score:.4f})")
    
    # Plot feature importances
    plt.figure(figsize=(10, 6))
    plt.title("Feature Importances")
    plt.bar(range(X.shape[1]), importances[indices], color="#3b82f6", align="center")
    plt.xticks(range(X.shape[1]), [fitur_kolom[i] for i in indices], rotation=45, ha='right')
    plt.xlim([-1, X.shape[1]])
    plt.tight_layout()
    plt.savefig(os.path.join(IMAGES_DIR, 'feature_importance.png'), dpi=150)
    plt.close()
    print("  Grafik Feature Importance berhasil disimpan.")
except Exception as e:
    print(f"  Gagal mengekstrak Feature Importance: {e}")




[TAHAP 10] Analisis Feature Importance...
--------------------------------------------------
  Feature ranking:
    1. ph (0.1299)
    2. Chloramines (0.1294)
    3. Solids (0.1213)
    4. Sulfate (0.1201)
    5. Hardness (0.1188)
    6. Organic_carbon (0.0993)
    7. Turbidity (0.0993)
    8. Trihalomethanes (0.0910)
    9. Conductivity (0.0909)
  Grafik Feature Importance berhasil disimpan.


In [ ]:
# TAHAP 11: SIMPAN ARTIFACTS


In [21]:
print("\n[TAHAP 11] Menyimpan Model & Metadata...")
print("-" * 50)

joblib.dump(final_model, os.path.join(MODELS_DIR, 'model.pkl'))
joblib.dump(scaler, os.path.join(MODELS_DIR, 'scaler.pkl'))

metadata = {
    'model_name': best_model_name,
    'optimal_threshold': float(best_thresh),
    'features': fitur_kolom,
    'feature_importances': feature_importances_dict,
    'dataset_size': int(len(df)),
    'evaluation': {
        'test_accuracy': round(float(best_thresh_acc), 4),
        'precision': round(float(best_thresh_prec), 4),
        'recall': round(float(best_thresh_rec), 4),
        'f1_score': round(float(best_thresh_f1), 4),
        'roc_auc': round(float(best_result['roc_auc']), 4)
    },
    'preprocessing': {
        'missing_value_strategy': 'KNNImputer (k=5)',
        'outlier_handling': 'IQR Capping',
        'oversampling': 'ADASYN',
        'scaling': 'StandardScaler',
        'class_weight': 'balanced'
    },
    'training_date': time.strftime('%Y-%m-%d %H:%M:%S')
}

with open(os.path.join(MODELS_DIR, 'training_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print("  Pipeline ADVANCED Selesai!\n")



[TAHAP 11] Menyimpan Model & Metadata...
--------------------------------------------------
  Pipeline ADVANCED Selesai!

